In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_COLAB = ROOT0 / "colab"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import *
from libs.dashcyto_lib import DASH_CYTO
from libs.calc_degs_lib import CALC_DEGS

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(


In [3]:
root0 = Path(dic_yml['root0'])
root0_data = Path(dic_yml['root0_data'])
root_colab = root0 / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"


root_project = root0_data / s_project
PSI_ID = 'TCGA-BRCA'
disease = PSI_ID

root_disease = root_project / PSI_ID

print(">>> root0", root0)
print(">>> root_colab", root_colab)
print(">>> root0_data", root0_data)
print(">>> root_project", root_project)
print(">>> root_disease", root_disease)


>>> root0 /home/flavio/uv/perturb_agent
>>> root_colab /home/flavio/uv/perturb_agent/colab
>>> root0_data /home/flavio/uv/perturb_agent/data
>>> root_project /home/flavio/uv/perturb_agent/data/TCGA
>>> root_disease /home/flavio/uv/perturb_agent/data/TCGA/TCGA-BRCA


In [4]:
root0 = Path(dic_yml['root0'])
root0_data = Path(dic_yml['root0_data'])
root_colab = root0 / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"


root_project = root0_data / s_project
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

root_project = create_dir(root0_data, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [5]:
s_omics

'RNA-Seq'

In [6]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0, root0_data=root0_data,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

Start opening tables ....
Building synonym dictionary ...

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC
>>> Tumor
>>> case Tumor
	DEGs 20006
		Up (#10358)
		Dw (#9648)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    12
1                      IG_C_pseudogene     4
2                            IG_D_gene     1
3                            IG_J_gene     9
4                      IG_J_pseudogene     1
5                            IG_V_gene   124
6                      IG_V_pseudogene    50
7                              Mt_tRNA    17
8                                  TEC   112
9                            TR_C_gene     5
10                           TR_D_gene     1
11                           TR_J_gene    10
12                           TR_V_gene    69
13                     TR_V_pseudogene     1
14                              lncRNA  3193
15                               miRNA   

In [7]:
from libs.tcga_gdc_lib import GDC

ROOT_DATA = root0 / "data"

gdc = GDC(ROOT0=ROOT0, ROOT_DATA0=ROOT_DATA)

### Get all programs

In [8]:
force = False
verbose = True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

prog_id = "TCGA"

df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)
print(len(df_psi))

df_psi.shape, df_psi.columns

File read at '/home/flavio/uv/perturb_agent/data/gdc_programs.txt'
Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/TCGA/primary_site_program_TCGA.tsv'
33


((33, 5),
 Index(['psi_id', 'primary_site', 'project_id', 'disease_type', 'name'], dtype='object'))

In [9]:
gdc.set_primary_site(psi_id=PSI_ID, verbose=True)


-----------------------------
>> root disease: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC
>> root samples: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC/samples
>> root lfc: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC/lfc
>> root mutations: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC/mutations
-----------------------------



True

In [10]:
verbose=False
df_tumor, df_normal, df_gtex_ctrl = gdc.get_file_expression_both_tumor_and_normal(verbose=verbose)

Dowloading normal files: 0.
Dowloading tumor files: 0..........10..........20..........30..........40..........50..........60..........70..........80..........90..........100..........110..........120..........130..........140..........150..........160..........170..........180..........190..........200....
Preparing GTEx metadata...
GTEx metadata prepared on df_meta_prep length: 7


In [11]:
# gdc.df_meta["SMTSD"].unique()

In [12]:
print(len(df_tumor.columns)-3)
df_tumor.head(3)

12


,geneid,symbol,biotype,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,tumor_12
0,ENSG00000000003,TSPAN6,protein_coding,2121,2582,218,1639,1087,544,788,1893,1198,816,1601,3864
1,ENSG00000000005,TNMD,protein_coding,1,0,0,1,0,0,2,1,0,0,35,1
2,ENSG00000000419,DPM1,protein_coding,1263,1301,327,1167,1123,686,1289,1446,1505,1744,1247,1299


In [13]:
print(len(df_normal.columns)-3)
df_normal.head(3)

1


,geneid,symbol,biotype,normal_1
0,ENSG00000000003,TSPAN6,protein_coding,1664
1,ENSG00000000005,TNMD,protein_coding,15
2,ENSG00000000419,DPM1,protein_coding,1274


In [14]:
print(len(df_gtex_ctrl.columns)-2)
df_gtex_ctrl.head(3)

3


,ensemblid,symbol,GTEX-SN8G-2126-SM-GPI8G,GTEX-T2IS-2326-SM-HL9TW,GTEX-Q2AG-2326-SM-EZ6KY
0,ENSG00000290825.2,DDX11L16,0,0,1
1,ENSG00000223972.6,DDX11L1,0,0,0
2,ENSG00000310526.1,WASH7P,433,373,515


### Check main cols

In [15]:
mtd.check_lfc_names(verbose=True)


Echo Parameters:

>>> lfc file exists: /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC/lfc/TCGA-CESC_final_LFC_Tumor_x_CTRL_not_normalized.tsv 

Checking columns:
	 ensembl_id ok
	 symbol ok
	 biotype ok
	 description ???
	 lfc ok
	 abs_lfc ok
	 pval ok
	 fdr ok




In [16]:
fname, fname_cutoff = mtd.set_enrichment_name()
fname, os.path.exists(os.path.join(mtd.root_enrich, fname)), fname_cutoff, os.path.exists(os.path.join(mtd.root_enrich, fname_cutoff))

('enricher_Reactome_Pathways_2024_TCGA-CESC_RNA-Seq_for_Tumor_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000.tsv',
 False,
 'enricher_Reactome_Pathways_2024_TCGA-CESC_RNA-Seq_for_Tumor_x_ctrl_not_normalized_cutoff_lfc_0.900_fdr_1.000_pathway_pval_0.050_fdr_0.050_num_genes_0.05.tsv',
 False)

In [17]:
try:
    dflfc_ori = mtd.dflfc_ori
    print(len(dflfc_ori))
except:
    dflfc_ori = pd.DataFrame()
    
dflfc_ori.head(3)

32213


,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
0,ENSG00000264232,LINC01916,lncRNA,-23.778,3.656,-6.505,NaN,NaN,9.331,DESeq2,23.778
1,ENSG00000179046,TRIML2,protein_coding,21.789,3.100,7.028,2.090e-12,4.906e-11,28.213,DESeq2,21.789
2,ENSG00000205325,AC005863.1,lncRNA,20.968,3.741,5.606,2.076e-08,2.626e-07,16.533,DESeq2,20.968


In [18]:
lista = ['lncRNA', 'LNC']
dflfc_lnc = dflfc_ori[dflfc_ori.biotype.isin(lista)]
print(f"There are {len(dflfc_lnc)} LNCs\n")
dflfc_lnc.tail(3)

There are 9276 LNCs



,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
32209,ENSG00000234696,GPR50-AS1,lncRNA,0.0,0.0,0.0,1.0,NaN,0.0,DESeq2,0.0
32210,ENSG00000235519,LINC01815,lncRNA,0.0,0.0,0.0,1.0,NaN,0.0,DESeq2,0.0
32212,ENSG00000248268,AC010275.1,lncRNA,0.0,0.0,0.0,1.0,NaN,0.0,DESeq2,0.0


In [19]:
dflfc_ori = mtd.dflfc_ori
print(len(dflfc_ori))
dflfc_ori_symb = dflfc_ori[~pd.isnull(dflfc_ori.symbol)]
print(len(dflfc_ori_symb))

32213
32213


### Unique symbols

In [20]:
try:
    symbols = np.unique(dflfc_ori.symbol)
except:
    symbols = []
    
len(symbols), len(dflfc_ori)

(32213, 32213)

In [21]:
try:
    dflfc = mtd.dflfc
    print(len(dflfc))
except:
    dflfc = pd.DataFrame()
    
dflfc.head(3)

20006


,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
0,ENSG00000258017,AC011603.2,lncRNA,9.994,0.309,32.338,2.046e-229,6.404e-225,20150.844,DESeq2,9.994
1,ENSG00000179818,PCBP1-AS1,lncRNA,11.946,0.464,25.728,5.683e-146,8.893e-142,8840.075,DESeq2,11.946
2,ENSG00000269243,AC008894.2,lncRNA,8.606,0.335,25.701,1.149e-145,1.199e-141,1939.699,DESeq2,8.606


In [22]:
dfbest = mtd.cfg.open_best_ptw_cutoff()
dfbest

""


In [23]:
want_see_best_cutoff = False

if want_see_best_cutoff:
    dfbest = mtd.cfg.dfbest_cutoffs
else:
    dfbest = pd.DataFrame()
dfbest    

""


In [24]:
if want_see_best_cutoff:
    dfbest = mtd.cfg.dfbest_cutoffs
    dfa = dfbest[(dfbest.case == case) & (dfbest.normalization == mtd.normalization) & (dfbest.geneset_num == geneset_num) ]
else:
    dfa = pd.DataFrame()

dfa

""


### Minimum LFC cutoff

In [25]:
np.log2(1.4)

0.48542682717024166

### DEGs simulation: no DEG/DAPs per cases
### Saving simulation file dfsim in config:
  - all_lfc_cutoffs_taubate_covid19.tsv

#### Sampling

### Cutoff sets to generate the statistical data
  - inf lfc cutoff: 0.40 --> 0.48 ~ 40% modulation  --> 0.65
  - sup fdr cutoff: 0.75 --> no more than

In [26]:
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
mtd.lfc_list = lfc_list
lfc_list[-1] = 0.0
lfc_list

array([1.   , 0.975, 0.95 , 0.925, 0.9  , 0.875, 0.85 , 0.825, 0.8  ,
       0.775, 0.75 , 0.725, 0.7  , 0.675, 0.65 , 0.625, 0.6  , 0.575,
       0.55 , 0.525, 0.5  , 0.475, 0.45 , 0.425, 0.4  , 0.375, 0.35 ,
       0.325, 0.3  , 0.275, 0.25 , 0.225, 0.2  , 0.175, 0.15 , 0.125,
       0.1  , 0.075, 0.05 , 0.025, 0.   ])

In [27]:
fdr_list = np.arange(0.05, 0.76, .01)
mtd.fdr_list = fdr_list
fdr_list

array([0.05, 0.06, 0.07, 0.08, 0.09, 0.1 , 0.11, 0.12, 0.13, 0.14, 0.15,
       0.16, 0.17, 0.18, 0.19, 0.2 , 0.21, 0.22, 0.23, 0.24, 0.25, 0.26,
       0.27, 0.28, 0.29, 0.3 , 0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37,
       0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48,
       0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.59,
       0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.7 ,
       0.71, 0.72, 0.73, 0.74, 0.75])

In [28]:
cutoff_list = np.round([(x, y) for x in lfc_list for y in fdr_list],3)
print(len(cutoff_list))
cutoff_list[:5], cutoff_list[-5:]

2911


(array([[1.  , 0.05],
        [1.  , 0.06],
        [1.  , 0.07],
        [1.  , 0.08],
        [1.  , 0.09]]),
 array([[0.  , 0.71],
        [0.  , 0.72],
        [0.  , 0.73],
        [0.  , 0.74],
        [0.  , 0.75]]))

### Saving simulations: calc_degs_cutoff_simulation()

  - config/all_lfc_cutoffs_medulloblastoma.tsv
  - while looping in case_list -> save_file -> save txt files

In [29]:
cutoff_list

array([[1.  , 0.05],
       [1.  , 0.06],
       [1.  , 0.07],
       ...,
       [0.  , 0.73],
       [0.  , 0.74],
       [0.  , 0.75]])

In [30]:
verbose=False
force=False

dfsim = mtd.calc_degs_cutoff_simulation(cutoff_list=cutoff_list, force=force, save_file=force, n_echo=-1, verbose=verbose)

print(dfsim.columns)
print(len(dfsim))

Index(['case', 'normalization', 'cutoff', 'LFC_cut', 'lfc_FDR_cut', 'degs', 'n_degs',
       'degs_ensembl', 'n_degs_ensembl', 'degs_up', 'n_degs_up', 'degs_up_ensembl',
       'n_degs_up_ensembl', 'degs_dw', 'n_degs_dw', 'degs_dw_ensembl', 'n_degs_dw_ensembl'],
      dtype='object')
2911


In [31]:
dfsim = dfsim.sort_values(['case', 'lfc_FDR_cut', 'LFC_cut'], ascending=[False, True, False])
dfsim.head(3)

,case,normalization,cutoff,LFC_cut,lfc_FDR_cut,degs,n_degs,degs_ensembl,n_degs_ensembl,degs_up,n_degs_up,degs_up_ensembl,n_degs_up_ensembl,degs_dw,n_degs_dw,degs_dw_ensembl,n_degs_dw_ensembl
0,Tumor,not_normalized,1.000 - 0.050,1.000,0.05,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",10993,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",10993,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5838,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5838,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5155,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5155
71,Tumor,not_normalized,0.975 - 0.050,0.975,0.05,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",11062,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",11062,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5883,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5883,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5179,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5179
142,Tumor,not_normalized,0.950 - 0.050,0.950,0.05,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",11114,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",11114,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5916,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5916,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5198,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5198


### Does the simulation worked?

In [32]:
dfsim = mtd.open_simulation_table()
print(len(dfsim))

dfsim2 = dfsim[dfsim.case == case]
dfsim2.head(3).T

2911


,0,1,2
case,Tumor,Tumor,Tumor
normalization,not_normalized,not_normalized,not_normalized
cutoff,1.000 - 0.050,1.000 - 0.060,1.000 - 0.070
LFC_cut,1.0,1.0,1.0
lfc_FDR_cut,0.05,0.06,0.07
degs,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
n_degs,10993,11400,11742
degs_ensembl,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
n_degs_ensembl,10993,11400,11742
degs_up,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...","['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...","['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA..."


In [33]:
LFC_cut = 1
lfc_FDR_cut = 0.05

# (dfsim.case == case) &
dfsim[ (dfsim.LFC_cut == LFC_cut) & (dfsim.lfc_FDR_cut == lfc_FDR_cut)].T

,0
case,Tumor
normalization,not_normalized
cutoff,1.000 - 0.050
LFC_cut,1.0
lfc_FDR_cut,0.05
degs,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
n_degs,10993
degs_ensembl,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
n_degs_ensembl,10993
degs_up,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA..."


In [34]:
np.unique(dfsim.LFC_cut)

array([0.   , 0.025, 0.05 , 0.075, 0.1  , 0.125, 0.15 , 0.175, 0.2  ,
       0.225, 0.25 , 0.275, 0.3  , 0.325, 0.35 , 0.375, 0.4  , 0.425,
       0.45 , 0.475, 0.5  , 0.525, 0.55 , 0.575, 0.6  , 0.625, 0.65 ,
       0.675, 0.7  , 0.725, 0.75 , 0.775, 0.8  , 0.825, 0.85 , 0.875,
       0.9  , 0.925, 0.95 , 0.975, 1.   ])

In [35]:
dfsim.LFC_cut.min(), dfsim.LFC_cut.max()

(0.0, 1.0)

In [36]:
np.unique(dfsim.lfc_FDR_cut)

array([0.05, 0.06, 0.07, 0.08, 0.09, 0.1 , 0.11, 0.12, 0.13, 0.14, 0.15,
       0.16, 0.17, 0.18, 0.19, 0.2 , 0.21, 0.22, 0.23, 0.24, 0.25, 0.26,
       0.27, 0.28, 0.29, 0.3 , 0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37,
       0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48,
       0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.59,
       0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.7 ,
       0.71, 0.72, 0.73, 0.74, 0.75])

In [37]:
dfsim.lfc_FDR_cut.min(), dfsim.lfc_FDR_cut.max(), 

(0.05, 0.75)

In [38]:
dfsim.LFC_cut.min(), dfsim.LFC_cut.max(), 

(0.0, 1.0)

In [39]:
dfsim.lfc_FDR_cut.min(), dfsim.lfc_FDR_cut.max(), 

(0.05, 0.75)

### Simulations

In [40]:
case_list

['Tumor']

In [41]:
case = 'Tumor'

mtd.open_case(case, verbose=False)

fname, fname_ori, title = mtd.set_lfc_names()
print(f"fname '{fname}' and title '{title}'")
print(f"LFC cutoff: lfc={mtd.LFC_cut:.3f} fdr={mtd.lfc_FDR_cut}")

print("")
print(mtd.echo_parameters())

fname 'TCGA-CESC_final_LFC_Tumor_x_CTRL_not_normalized.tsv' and title 'Tumor (not_normalized)'
LFC cutoff: lfc=0.900 fdr=1

	20006/20006 DEGs/ensembl.
		Up 10358/10358 DEGs/ensembl.
		Dw 9648/9648 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


### DEGs/DAPs frequency
### Not Normalized

In [42]:
#dfsim = pdreadcsv( mtd.cfg.fname_lfc_cutoff, mtd.cfg.root_config)
dfsim = mtd.cfg.open_all_lfc_cutoff()
print(len(dfsim))
dfsim.tail(2)

2911


,case,normalization,cutoff,LFC_cut,lfc_FDR_cut,degs,n_degs,degs_ensembl,n_degs_ensembl,degs_up,n_degs_up,degs_up_ensembl,n_degs_up_ensembl,degs_dw,n_degs_dw,degs_dw_ensembl,n_degs_dw_ensembl
2909,Tumor,not_normalized,0.000 - 0.740,0.0,0.74,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",26621,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",26621,"['5S_rRNA', 'A1BG-AS1', 'A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'A3GALT2', 'AACS', ...",13804,"['5S_rRNA', 'A1BG-AS1', 'A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'A3GALT2', 'AACS', ...",13804,"['A1BG', 'A2M', 'A4GNT', 'AAAS', 'AADAT', 'AARSD1', 'AARSD1P1', 'AASDH', 'AA...",12817,"['A1BG', 'A2M', 'A4GNT', 'AAAS', 'AADAT', 'AARSD1', 'AARSD1P1', 'AASDH', 'AA...",12817
2910,Tumor,not_normalized,0.000 - 0.750,0.0,0.75,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",26768,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",26768,"['5S_rRNA', 'A1BG-AS1', 'A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'A3GALT2', 'AACS', ...",13870,"['5S_rRNA', 'A1BG-AS1', 'A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'A3GALT2', 'AACS', ...",13870,"['A1BG', 'A2M', 'A4GNT', 'AAAS', 'AADAT', 'AARSD1', 'AARSD1P1', 'AASDH', 'AA...",12898,"['A1BG', 'A2M', 'A4GNT', 'AAAS', 'AADAT', 'AARSD1', 'AARSD1P1', 'AASDH', 'AA...",12898


In [43]:
i=0
case = case_list[i]
print(">>>", case)
df2 = dfsim[dfsim.case == case].copy()
print(len(df2))
df2.head(2)

>>> Tumor
2911


,case,normalization,cutoff,LFC_cut,lfc_FDR_cut,degs,n_degs,degs_ensembl,n_degs_ensembl,degs_up,n_degs_up,degs_up_ensembl,n_degs_up_ensembl,degs_dw,n_degs_dw,degs_dw_ensembl,n_degs_dw_ensembl
0,Tumor,not_normalized,1.000 - 0.050,1.0,0.05,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",10993,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",10993,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5838,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5838,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5155,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5155
1,Tumor,not_normalized,1.000 - 0.060,1.0,0.06,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",11400,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",11400,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",6050,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",6050,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5350,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5350


In [44]:
dfsim = mtd.open_simulation_table()
print(len(dfsim))
dfsim.head(2)

2911


,case,normalization,cutoff,LFC_cut,lfc_FDR_cut,degs,n_degs,degs_ensembl,n_degs_ensembl,degs_up,n_degs_up,degs_up_ensembl,n_degs_up_ensembl,degs_dw,n_degs_dw,degs_dw_ensembl,n_degs_dw_ensembl
0,Tumor,not_normalized,1.000 - 0.050,1.0,0.05,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",10993,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",10993,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5838,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",5838,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5155,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5155
1,Tumor,not_normalized,1.000 - 0.060,1.0,0.06,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",11400,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...",11400,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",6050,"['A2M-AS1', 'A2ML1-AS1', 'A2MP1', 'AACSP1', 'AAGAB', 'AATF', 'ABALON', 'ABCA...",6050,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5350,"['A1BG', 'AARSD1', 'AASS', 'ABCA10', 'ABCA2', 'ABCA5', 'ABCA6', 'ABCA8', 'AB...",5350


## Calc all Spearman Correlations - filter the 5 best not repeated fdrs
#### Plot abs_LFC x num of DEG/DAPs
#### corr_cutoff = -.90
#### calc corelation with mtd.LFC_cut_inf = 0.40

In [45]:
want_calc = False
corr_cutoff=-0.90
nregs_fdr = 5

force=False
verbose=False

df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'LFC_cut'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, force=force, verbose=verbose)

In [46]:
df_all_fdr.head(2)

,case,fdr,corr,first,chosen,label,method,n_degs_min,n_degs_max,LFC_cut_inf,degs_min,degs_max
0,Tumor,0.05,-1.0,True,True,fdr=0.05 corr=-1.000 ***,spearman,10993,11597,0.4,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
1,Tumor,0.06,-1.0,False,True,fdr=0.06 corr=-1.000,spearman,11400,12099,0.4,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."


In [47]:
print(len(df_all_fdr))
print(df_all_fdr.columns)

df_all_fdr[~pd.isnull(df_all_fdr['corr']) &  (df_all_fdr['corr'] <= -0.9)].head(2)

5
Index(['case', 'fdr', 'corr', 'first', 'chosen', 'label', 'method', 'n_degs_min', 'n_degs_max',
       'LFC_cut_inf', 'degs_min', 'degs_max'],
      dtype='object')


,case,fdr,corr,first,chosen,label,method,n_degs_min,n_degs_max,LFC_cut_inf,degs_min,degs_max
0,Tumor,0.05,-1.0,True,True,fdr=0.05 corr=-1.000 ***,spearman,10993,11597,0.4,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
1,Tumor,0.06,-1.0,False,True,fdr=0.06 corr=-1.000,spearman,11400,12099,0.4,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."


In [48]:
df_all_fdr[~pd.isna(df_all_fdr['corr']) &  (df_all_fdr['corr'] <= -0.9)]['corr'].min()

-1.0

### Plot all dfsim

In [49]:
# !pip3 install -U kaleido

In [50]:
fdr_list

array([0.05, 0.06, 0.07, 0.08, 0.09, 0.1 , 0.11, 0.12, 0.13, 0.14, 0.15,
       0.16, 0.17, 0.18, 0.19, 0.2 , 0.21, 0.22, 0.23, 0.24, 0.25, 0.26,
       0.27, 0.28, 0.29, 0.3 , 0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37,
       0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48,
       0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.59,
       0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.7 ,
       0.71, 0.72, 0.73, 0.74, 0.75])

In [51]:
for case in case_list:
    print(">>>", case)
    dic_fig = mtd.plot_all_dfsim(dfsim, case=case, fdr_list=fdr_list, width=1100, height=700, title=None, verbose=force)
        
    for key, fig in dic_fig.items():
        print("\t", key)
        fig.show()
        break # remove to see Up and Dw

    print("\n")
    

>>> Tumor
	 deg


In [52]:
# dfsim.columnscutoff_list

### Restricting the best fdr by Spearman's Correlation

### Must calc for each LFC_cut_inf

In [53]:
corr_cutoff=-0.90
nregs_fdr = 10
mtd.LFC_cut_inf = 0

verbose=False
force = False

'''
    calc_all_LFC_FDR_cutoffs:
        for case_list
            call calc_nDEG_curve_per_LFC_FDR()
'''
df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'LFC_cut'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                          force=force, verbose=verbose)
print(len(df_all_fdr))

10


### LFC_cut_inf = 0.4

In [54]:
LFC_cut_inf

0.4

In [55]:
dfsim = mtd.dfsim[mtd.dfsim.case == case]
dfsim = dfsim.sort_values(['lfc_FDR_cut', 'LFC_cut'], ascending=[True, False])

dfsim.lfc_FDR_cut.unique(), dfsim.LFC_cut.unique()

(array([0.05, 0.06, 0.07, 0.08, 0.09, 0.1 , 0.11, 0.12, 0.13, 0.14, 0.15,
        0.16, 0.17, 0.18, 0.19, 0.2 , 0.21, 0.22, 0.23, 0.24, 0.25, 0.26,
        0.27, 0.28, 0.29, 0.3 , 0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37,
        0.38, 0.39, 0.4 , 0.41, 0.42, 0.43, 0.44, 0.45, 0.46, 0.47, 0.48,
        0.49, 0.5 , 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.59,
        0.6 , 0.61, 0.62, 0.63, 0.64, 0.65, 0.66, 0.67, 0.68, 0.69, 0.7 ,
        0.71, 0.72, 0.73, 0.74, 0.75]),
 array([1.   , 0.975, 0.95 , 0.925, 0.9  , 0.875, 0.85 , 0.825, 0.8  ,
        0.775, 0.75 , 0.725, 0.7  , 0.675, 0.65 , 0.625, 0.6  , 0.575,
        0.55 , 0.525, 0.5  , 0.475, 0.45 , 0.425, 0.4  , 0.375, 0.35 ,
        0.325, 0.3  , 0.275, 0.25 , 0.225, 0.2  , 0.175, 0.15 , 0.125,
        0.1  , 0.075, 0.05 , 0.025, 0.   ]))

### For FDR == 0.05 (default cutoff) - there is no correlation, is a horizontal flat line for 0.05

In [56]:
mtd.LFC_cut_inf

0

In [57]:
fdr = 0.05

i=0
case = case_list[i]
print(">>>", case)

dfsim2 = dfsim[ (dfsim.case==case) & (dfsim.lfc_FDR_cut == fdr) & (dfsim.LFC_cut >= mtd.LFC_cut_inf) ]
len(dfsim2)

>>> Tumor


41

In [58]:
cols2=['case', 'n_degs', 'lfc_FDR_cut', 'LFC_cut']
dfsim2[cols2].head(5)

,case,n_degs,lfc_FDR_cut,LFC_cut
0,Tumor,10993,0.05,1.000
71,Tumor,11062,0.05,0.975
142,Tumor,11114,0.05,0.950
213,Tumor,11169,0.05,0.925
284,Tumor,11219,0.05,0.900


In [59]:
dfsim2[cols2].tail(5)

,case,n_degs,lfc_FDR_cut,LFC_cut
2556,Tumor,11597,0.05,0.100
2627,Tumor,11597,0.05,0.075
2698,Tumor,11597,0.05,0.050
2769,Tumor,11597,0.05,0.025
2840,Tumor,11597,0.05,0.000


In [60]:
dfsim3 = dfsim2.reset_index()

cols3=['index', 'n_degs', 'lfc_FDR_cut', 'LFC_cut']

dfsim3[cols3].head(2)

,index,n_degs,lfc_FDR_cut,LFC_cut
0,0,10993,0.05,1.000
1,71,11062,0.05,0.975


In [61]:
dfsim3[cols3].iloc[:, [0,1]].head(3)

,index,n_degs
0,0,10993
1,71,11062
2,142,11114


In [62]:
method='spearman'
corr = dfsim3[cols3].iloc[:, [0,1]].corr(method=method)
corr

,index,n_degs
index,1.000,0.964
n_degs,0.964,1.000


In [63]:
pd.isnull(corr)

,index,n_degs
index,False,False
n_degs,False,False


### mtd.LFC_cut_inf = 0.40

In [64]:
nregs_fdr = 10

LFC_cut_inf = dic_yml['LFC_cut_inf']
mtd.LFC_cut_inf = LFC_cut_inf

verbose=False
force = False

'''
    calc_all_LFC_FDR_cutoffs:
        for case_list
            call calc_nDEG_curve_per_LFC_FDR()
'''
df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'LFC_cut'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                          force=force, verbose=verbose)
print(len(df_all_fdr))

5


### WNT - Spearman

In [65]:
i=0
case = case_list[i]

df2 = df_all_fdr[ (df_all_fdr.case == case) ]
print(len(df2))

df2 = df_all_fdr[ (df_all_fdr.case == case) & ( pd.notnull(df_all_fdr['corr'])  ) ]
print(len(df2))
print(f"{mtd.s_deg_dap} max: {df2.n_degs_max.max()}")
df2.head(2)

5
5
DEG max: 13317


,case,fdr,corr,first,chosen,label,method,n_degs_min,n_degs_max,LFC_cut_inf,degs_min,degs_max
0,Tumor,0.05,-1.0,True,True,fdr=0.05 corr=-1.000 ***,spearman,10993,11597,0.4,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
1,Tumor,0.06,-1.0,False,True,fdr=0.06 corr=-1.000,spearman,11400,12099,0.4,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."


### G4 Spearman

In [66]:
for case in case_list:
    df2 = df_all_fdr[ (df_all_fdr.case == case) & ( pd.notnull(df_all_fdr['corr'])  ) ]
    print(f"case: {case:4} n={len(df2)} max {mtd.s_deg_dap}: {df2.n_degs_max.max()}")

case: Tumor n=5 max DEG: 13317


### Plot abs_LFC x num of DEGs/DAPs
  - set LFC_cut_inf

In [67]:
corr_cutoff, nregs_fdr, case_list

(-0.9, 10, ['Tumor'])

In [68]:
i=0
case  = case_list[i]
print(">>>", case)

cols2=['n_degs', 'LFC_cut']
limit_fdr = -1
method='spearman'

ret, dic_return = mtd.calc_nDEG_curve_per_LFC_FDR(case=case, cols2=cols2, 
                                                  corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                                  method=method, verbose=verbose)

len(dic_return)

>>> Tumor


3

In [69]:
list(dic_return.keys())

['df_fdr', 'name_list', 'fdrs']

In [70]:
len(dic_return['df_fdr'])

5

In [71]:
len(dic_return['name_list']), dic_return['name_list']

(5,
 ['fdr=0.05 corr=-1.000 ***',
  'fdr=0.06 corr=-1.000',
  'fdr=0.07 corr=-1.000',
  'fdr=0.08 corr=-1.000',
  'fdr=0.09 corr=-1.000'])

In [72]:
len(dic_return['fdrs']), np.array(dic_return['fdrs'])

(5, array([0.05, 0.06, 0.07, 0.08, 0.09]))

In [73]:
df_fdr = dic_return['df_fdr']
df_fdr.head(2)

,case,fdr,corr,first,chosen,label,method,n_degs_min,n_degs_max,LFC_cut_inf,degs_min,degs_max
0,Tumor,0.05,-1.0,True,True,fdr=0.05 corr=-1.000 ***,spearman,10993,11597,0.4,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
1,Tumor,0.06,-1.0,False,True,fdr=0.06 corr=-1.000,spearman,11400,12099,0.4,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."


In [74]:
LFC_cut_inf2 = mtd.LFC_cut_inf
corr_cutoff

-0.9

In [75]:
mtd.LFC_cut_inf = 0.
verbose = False

i=0
case = case_list[i]
mtd.open_case(case)
print(">>>", case)

ret, dic_fig, df_fdr = mtd.plot_nDEG_curve_per_LFC_FDR(case, width=1100, height=700, title=None, 
                                                       corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, verbose=verbose)

for key, fig in dic_fig.items():
    print(key)
    fig.show()

>>> Tumor
deg


up


down


## LFC_cut_inf = 0.4 - see the curve

### Ploting only Spearman's limiar curves

In [76]:

dic_fig = mtd.plot_all_LFC_FDR_cutoffs(width=1100, height=700, title=None, 
                                       corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, verbose=force)

for case in case_list:
    print(">>>", case)
    try:
        dic2 = dic_fig[case]
    except:
        continue
        
    for key, fig in dic2.items():
        fig.show()
        # do not show Up and Down
        break
    print("")

>>> Tumor


In [77]:
LFC_cut_inf = mtd.LFC_cut_inf
LFC_cut_inf

0.0

In [78]:
case = case_list[0]

df_all_fdr = mtd.open_fdr_lfc_correlation(case, LFC_cut_inf)
df2 = df_all_fdr[ pd.notnull(df_all_fdr['corr']) ]
print(len(df2))
df2.head(6)

10


,case,fdr,corr,first,chosen,label,method,n_degs_min,n_degs_max,LFC_cut_inf,degs_min,degs_max
0,Tumor,0.05,-0.964,True,True,fdr=0.05 corr=-0.964 ***,spearman,10993,11597,0,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
1,Tumor,0.06,-0.964,False,True,fdr=0.06 corr=-0.964,spearman,11400,12099,0,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
2,Tumor,0.07,-0.964,False,True,fdr=0.07 corr=-0.964,spearman,11742,12544,0,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
3,Tumor,0.08,-0.975,False,True,fdr=0.08 corr=-0.975,spearman,12060,12938,0,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
4,Tumor,0.09,-0.975,False,True,fdr=0.09 corr=-0.975,spearman,12371,13319,0,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."
5,Tumor,0.1,-0.975,False,True,fdr=0.10 corr=-0.975,spearman,12659,13690,0,"['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5...","['AC011603.2', 'PCBP1-AS1', 'AC008894.2', 'NME2', 'NELFCD', 'AHSA2P', 'PRMT5..."


### Summary DEG/DAPs + Up and Down (pre-best cutoff)

In [79]:
verbose=False
per_biotype= False
ensembl = False

dfa = mtd.summary_degs_up_down(per_biotype=per_biotype, ensembl=ensembl, verbose=verbose)
print(len(dfa))
dfa

7


case,Tumor
tot_measured,32213
n_degs,20006
n_degs_up,10358
n_degs_up_ensembl,10358
n_degs_dw,9648
n_degs_dw_ensembl,9648
n_ptw,0


### per_biotype

In [80]:
verbose=False
per_biotype= True
ensembl = False

dfa = mtd.summary_degs_up_down(per_biotype=per_biotype, ensembl=ensembl, verbose=verbose)
print(len(dfa))
dfa

52


case,Tumor
tot_measured,32213
IG_C_gene_up,12
IG_C_pseudogene_up,4
IG_D_gene_up,1
IG_J_gene_up,9
IG_J_pseudogene_up,1
IG_V_gene_up,124
IG_V_pseudogene_up,50
Mt_tRNA_up,17
TEC_up,112


### Barplot per per_biotype

In [81]:
per_biotype = True
ensembl = True
before_best_cutoff = True
fig, dfa = mtd.barplot_up_down_genes_per_case(per_biotype=per_biotype, ensembl=ensembl, before_best_cutoff=before_best_cutoff, width=1100, height=700, verbose=False)
if fig: fig.show()

In [82]:
width = 1300
          
fig = mtd.plot_all_degs_up_down_per_cutoffs(width=width, height=600, title='', y_anchor=1.05, color_up='darkred', color_dw='navy', plot_bgcolor='whitesmoke', verbose=False)
if fig: fig.show()

ValueError: Invalid property specified for object of type plotly.graph_objs.Layout: 'xaxis2'

Did you mean "xaxis"?

    Valid properties:
        activeselection
            :class:`plotly.graph_objects.layout.Activeselection`
            instance or dict with compatible properties
        activeshape
            :class:`plotly.graph_objects.layout.Activeshape`
            instance or dict with compatible properties
        annotations
            A tuple of
            :class:`plotly.graph_objects.layout.Annotation`
            instances or dicts with compatible properties
        annotationdefaults
            When used in a template (as
            layout.template.layout.annotationdefaults), sets the
            default property values to use for elements of
            layout.annotations
        autosize
            Determines whether or not a layout width or height that
            has been left undefined by the user is initialized on
            each relayout. Note that, regardl

### Simple summary

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=False, ensembl=False, verbose=False)
dfa

### per Biotype

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=True, ensembl=False, verbose=False)
dfa

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=True, ensembl=True, verbose=False)
dfa

### Review data

In [ ]:
want_review_data = True

if want_review_data:
    i=0
    case = case_list[i]
    mtd.open_case(case, verbose=False)
    
    fname, fname_ori, title = mtd.set_lfc_names()
    print(f"fname '{fname}' and title '{title}'")
    print(f"LFC cutoff: lfc={mtd.LFC_cut:.3f} fdr={mtd.lfc_FDR_cut}")
    
    print("")
    mtd.echo_parameters()

### LNCs

In [ ]:
lista = ['lncRNA', 'LNC']
dflfc_lnc = dflfc_ori[dflfc_ori.biotype.isin(lista)]
print(len(dflfc_lnc))
dflfc_lnc.tail(3)

In [ ]:
np.unique(dflfc_lnc.biotype)